In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

In [7]:
from langchain_ollama import ChatOllama

try:
    llm = ChatOllama(
        model="llama3",
        temperature=0,
    )
except Exception as e:
    print(f"Error initializing language model: {e}")
    llm = None

In [12]:
# --- Define Simulated Sub-Agent Handlers ---

def booking_handler(request: str) -> str:
    """Simulates the Booking Agent handling a request."""
    print("\n--- DELEGATING TO BOOKING HANDLER ---")
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."

def info_handler(request: str) -> str:
    """Simulates the Info Agent handling a request."""
    print("\n--- DELEGATING TO INFO HANDLER ---")
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."

def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delegated."""
    print("\n--- HANDLING UNCLEAR REQUEST ---")
    return f"Coordinator could not delegate request: '{request}'. Please clarify."

# --- Define Coordinator Router Chain (equivalent to ADK coordinator's instruction) ---
# This chain decides which handler to delegate to.
coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the user's request and determine which specialist handler should process it.
     - If the request is related to booking flights or hotels, output 'booker'.
     - For all other general information questions, output 'info'.
     - If the request is unclear or doesn't fit either category, output 'unclear'.
     ONLY output one word: 'booker', 'info', or 'unclear'."""),
    ("user", "{request}")
])

if llm:
    coordinator_router_chain = coordinator_router_prompt | llm | StrOutputParser()

# --- Define the delegation logic ---
# Use RunnableBranch to route based on the router chain's output.

# Define th branches for the RunnableBranch
branches = {
    "booker": RunnablePassthrough.assign(output= lambda x:
                                        booking_handler(x['request']['request'])),
    "info": RunnablePassthrough.assign(output= lambda x: 
                                        info_handler(x['request']['request'])),
    "unclear": RunnablePassthrough.assign(output=lambda x: 
                                        unclear_handler(x['request']['request'])),
}

# Create RunnableBranch. It takes the output of the router chain and routes the original input ('request') to the corresponding handler.
delegation_branch = RunnableBranch(
    (lambda x: x['decision'].strip() == 'booker', branches["booker"]),
    (lambda x: x['decision'].strip() == 'info', branches["info"]),
    branches["unclear"]
)

# Combine the router chain and the delegation branch into a single runnable
# The router chain's output ('decision') is passed along with the original input ('request') to the delegation branch.
coordinator_agent = {
    "decision": coordinator_router_chain,
    "request": RunnablePassthrough()
} | delegation_branch | (lambda x: x['output'])  # Extract the final output

In [9]:
# --- Example Usage ---
def main():
    if not llm:
        print("\nSkipping execution due to LLM initialiation failure.")
        return

    print("--- Running with a booking request ---")
    request_a = "Book me a flight to Barcelona."
    result_a = coordinator_agent.invoke({"request": request_a})
    print(f"Final result A: {result_a}")

    print("--- Running with an info request ---")
    request_b = "What is the capital of Catalonia?"
    result_b = coordinator_agent.invoke({"request": request_b})
    print(f"Final result B: {result_b}")

    print("--- Running with an unclear request ---")
    request_c = "Explain me the relativity equation."
    result_c = coordinator_agent.invoke({"request": request_c})
    print(f"Final result C: {result_c}")

In [13]:
if __name__== "__main__":
    main()

--- Running with a booking request ---

--- DELEGATING TO BOOKING HANDLER ---
Final result A: Booking Handler processed request: 'Book me a flight to Barcelona.'. Result: Simulated booking action.
--- Running with an info request ---

--- DELEGATING TO INFO HANDLER ---
Final result B: Info Handler processed request: 'What is the capital of Catalonia?'. Result: Simulated information retrieval.
--- Running with an unclear request ---

--- DELEGATING TO INFO HANDLER ---
Final result C: Info Handler processed request: 'Explain me the relativity equation.'. Result: Simulated information retrieval.
